# E-Commerce Customer Analytics

## Customer Segmentation using RFM Analysis and KMeans Clustering

This project analyzes transactional e-commerce data to understand customer purchasing behavior and identify valuable customer segments.

### Objectives

- Analyze sales performance
- Identify top-selling products
- Explore customer purchasing behavior
- Perform customer segmentation using RFM analysis
- Apply KMeans clustering to identify customer groups

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_style("whitegrid")

# Load Dataset

In [2]:
df = pd.read_csv("../data/processed/online_retail_clean.csv")

df.head()

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,total_price,year,month
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12


# Dataset Overview

The dataset contains transactional data from an online retail store between 2010 and 2011.

Each row represents a product purchased within an invoice.

Main features include:

- Invoice number
- Product information
- Quantity purchased
- Transaction date
- Customer ID
- Country
- Revenue generated

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

# Revenue Analysis

This section explores revenue trends over time to better understand sales performance and customer purchasing behavior.

In [ ]:
monthly_sales = (
    df.groupby(["year", "month"])["total_price"]
    .sum()
    .reset_index()
)

monthly_sales["date"] = pd.to_datetime(
    monthly_sales["year"].astype(str)
    + "-"
    + monthly_sales["month"].astype(str)
)

monthly_sales.head()

In [ ]:
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=monthly_sales,
    x="date",
    y="total_price",
    marker="o"
)

plt.title("Monthly Revenue Trend", fontsize=16)
plt.xlabel("Date")
plt.ylabel("Revenue")

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

## Insights

- Revenue shows a strong upward trend throughout the year.
- Sales activity increases significantly during later months.
- This behavior may reflect seasonality effects and holiday shopping patterns.

## Insights

- Revenue shows a strong upward trend throughout the year.
- Sales activity increases significantly during later months.
- This behavior may reflect seasonality effects and holiday shopping patterns.

In [ ]:
top_products = (
    df.groupby("description")["total_price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    x=top_products.values,
    y=top_products.index
)

plt.title("Top 10 Products by Revenue", fontsize=16)
plt.xlabel("Revenue")
plt.ylabel("Product")

plt.tight_layout()

plt.show()

## Insights

- A small group of products generates a significant portion of total revenue.
- Several top-performing products are related to home decoration and gift items.
- Product concentration suggests opportunities for inventory and marketing optimization.

## Insights

- A small group of products generates a significant portion of total revenue.
- Several top-performing products are related to home decoration and gift items.
- Product concentration suggests opportunities for inventory and marketing optimization.

In [ ]:
top_countries = (
    df.groupby("country")["total_price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_countries

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    x=top_countries.values,
    y=top_countries.index
)

plt.title("Top 10 Countries by Revenue", fontsize=16)
plt.xlabel("Revenue")
plt.ylabel("Country")

plt.tight_layout()

plt.show()

## Insights

- The United Kingdom dominates total revenue by a large margin.
- Several European countries contribute significantly to sales.
- The dataset suggests a strong concentration of customers within specific geographic regions.

# Customer Segmentation

To better understand customer behavior, we apply RFM Analysis and KMeans clustering.

## RFM Metrics

- Recency → How recently a customer purchased
- Frequency → How often a customer purchases
- Monetary → How much the customer spends

In [ ]:
from datetime import timedelta

In [ ]:
snapshot_date = df["invoicedate"].max()

snapshot_date

In [ ]:
rfm = df.groupby("customer_id").agg({
    "invoicedate": lambda x: (snapshot_date - x.max()).days,
    "invoice": "nunique",
    "total_price": "sum"
})

rfm.columns = ["recency", "frequency", "monetary"]

rfm.head()

## RFM Summary Statistics

## Insights

- Most customers purchase infrequently.
- A small number of customers generate very high revenue.
- Customer behavior appears highly skewed, which is common in e-commerce datasets.

# Data Preprocessing

Before applying KMeans clustering, we preprocess the RFM features.

Steps:

- Remove invalid values
- Apply log transformation
- Standardize features

In [ ]:
rfm = rfm[rfm["monetary"] > 0]

rfm.head()

In [ ]:
rfm_log = np.log1p(rfm)

rfm_log.head()

In [ ]:
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm_log)

In [ ]:
rfm_scaled[:5]

## Why preprocessing matters

Customer purchase behavior is highly skewed.

Without normalization, customers with extremely large purchases would dominate the clustering process.

Log transformation and scaling help create more balanced clusters.

## Why preprocessing matters

Customer purchase behavior is highly skewed.

Without normalization, customers with extremely large purchases would dominate the clustering process.

Log transformation and scaling help create more balanced clusters.

In [ ]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

kmeans.fit(rfm_scaled)

In [ ]:
rfm["cluster"] = kmeans.labels_

rfm.head()

# Cluster Summary

In [ ]:
cluster_summary = rfm.groupby("cluster").agg({
    "recency": "mean",
    "frequency": "mean",
    "monetary": "mean"
})

cluster_summary

## Cluster Interpretation

- Some customer groups show low frequency and low spending behavior.
- Other segments represent loyal and high-value customers.
- A small group of customers contributes disproportionately to revenue.

# Customer Segment Visualization

In [ ]:
plt.figure(figsize=(12, 7))

sns.scatterplot(
    data=rfm,
    x="frequency",
    y="monetary",
    hue="cluster",
    palette="Set2",
    s=100
)

plt.title("Customer Segments", fontsize=16)
plt.xlabel("Frequency")
plt.ylabel("Monetary")

plt.yscale("log")

plt.tight_layout()

plt.show()

## Final Insights

- Most customers are low-frequency buyers.
- A smaller segment contains highly valuable and loyal customers.
- The distribution of spending behavior is highly uneven.
- Customer segmentation can support targeted marketing and retention strategies.